<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/10.%20Clustering%20Techniques/ch10_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| Informasi | Keterangan |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |
| **Institusi** | Telkom University |

---


# Chapter 10: Clustering Techniques

## Unsupervised Learning: Clustering

Clustering adalah teknik **unsupervised learning** yang mengelompokkan data ke dalam cluster berdasarkan kemiripan, tanpa label yang telah ditentukan sebelumnya.

### Aplikasi Clustering
- Customer segmentation
- Anomaly detection
- Document grouping / topic discovery
- Image segmentation
- Genomics: gene expression grouping

## K-Means Clustering

K-Means mempartisi $n$ data ke $K$ cluster dengan meminimalkan **Within-Cluster Sum of Squares (WCSS)**:

$$\min_{\mathbf{c}_1,...,\mathbf{c}_K} \sum_{k=1}^{K} \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \mathbf{c}_k\|^2$$

di mana $\mathbf{c}_k$ adalah centroid cluster ke-$k$.

**Algoritma K-Means (Lloyd's algorithm)**:
1. Inisialisasi K centroid secara acak (atau dengan K-Means++)
2. **Assignment step**: assign setiap titik ke centroid terdekat
3. **Update step**: hitung ulang centroid dari anggota cluster
4. Ulangi langkah 2-3 hingga konvergen

### Pemilihan K: Elbow Method

Plot inertia (WCSS) vs $K$; cari titik "siku" di mana pengurangan inertia mulai melambat signifikan.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons, load_iris, load_wine
from sklearn.cluster import (
    KMeans, MiniBatchKMeans, DBSCAN,
    AgglomerativeClustering, SpectralClustering
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage

# Generate datasets
X_blobs, y_blobs = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=42)
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)

# Load real datasets
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

print(f'Blobs: {X_blobs.shape}')
print(f'Moons: {X_moons.shape}')
print(f'Iris:  {X_iris.shape}')


In [ ]:
# Elbow Method for optimal K
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_blobs)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_blobs, km.labels_)
    silhouette_scores.append(sil)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Elbow curve
axes[0].plot(K_range, inertias, 'b-o', lw=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method: Inertia vs K')
best_elbow_k = K_range[np.argmax(np.diff(np.diff(inertias)))+1]
axes[0].axvline(x=best_elbow_k, color='red', linestyle='--', label=f'Elbow at K={best_elbow_k}')
axes[0].legend()

# Silhouette score
axes[1].plot(K_range, silhouette_scores, 'g-s', lw=2, markersize=8)
best_sil_k = K_range[np.argmax(silhouette_scores)]
axes[1].axvline(x=best_sil_k, color='red', linestyle='--', label=f'Best K={best_sil_k}')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs K')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Best K by Elbow: {best_elbow_k}')
print(f'Best K by Silhouette: {best_sil_k} (score={max(silhouette_scores):.4f})')


In [ ]:
# K-Means clustering visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cmaps = ['Set1', 'Set2', 'tab10']

for ax, k in zip(axes, [2, 4, 6]):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_blobs)
    sil = silhouette_score(X_blobs, labels)
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='tab10', s=20, alpha=0.7)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               marker='*', s=300, c='red', edgecolors='black', linewidth=1.5, label='Centroids')
    ax.set_title(f'K-Means K={k}\nInertia={km.inertia_:.0f}, Silhouette={sil:.3f}')
    ax.legend()

plt.tight_layout()
plt.show()


## 10.2 Silhouette Analysis

Silhouette coefficient mengukur seberapa mirip setiap titik dengan clusternya sendiri dibandingkan cluster lain:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

di mana:
- $a(i)$ = rata-rata jarak titik $i$ ke titik lain dalam cluster yang sama (intra-cluster distance)
- $b(i)$ = rata-rata jarak titik $i$ ke titik dalam cluster terdekat yang bukan clusternya

$s(i) \in [-1, 1]$: nilai mendekati 1 berarti cluster assignment sangat baik.


In [ ]:
from sklearn.metrics import silhouette_samples

# Silhouette plot for K=4
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels4 = km4.fit_predict(X_blobs)
sil_samples = silhouette_samples(X_blobs, labels4)
mean_sil = silhouette_score(X_blobs, labels4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Silhouette plot
y_lower = 10
for k in range(4):
    sil_k = np.sort(sil_samples[labels4 == k])
    size_k = sil_k.shape[0]
    y_upper = y_lower + size_k
    color = plt.cm.tab10(k / 4)
    ax1.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_k, facecolor=color, alpha=0.7)
    ax1.text(-0.05, y_lower + 0.5 * size_k, f'Cluster {k}')
    y_lower = y_upper + 5

ax1.axvline(x=mean_sil, color='red', linestyle='--', label=f'Mean={mean_sil:.3f}')
ax1.set_title('Silhouette Plot - K-Means K=4')
ax1.set_xlabel('Silhouette Coefficient')
ax1.set_ylabel('Cluster')
ax1.legend()

# Cluster scatter
ax2.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels4, cmap='tab10', s=15, alpha=0.7)
ax2.scatter(km4.cluster_centers_[:, 0], km4.cluster_centers_[:, 1],
            marker='*', s=250, c='black', zorder=5)
ax2.set_title(f'K-Means K=4 (Silhouette={mean_sil:.3f})')

plt.tight_layout()
plt.show()


## 10.3 DBSCAN: Density-Based Clustering

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) mengelompokkan titik-titik yang saling berdekatan dalam ruang, tanpa perlu menentukan jumlah cluster.

Parameter:
- **eps** ($\varepsilon$): radius neighborhood
- **min_samples**: jumlah minimum titik dalam radius $\varepsilon$ untuk membentuk dense region

Tipe titik:
- **Core point**: memiliki $\geq$ min_samples titik dalam radius $\varepsilon$
- **Border point**: dalam radius $\varepsilon$ dari core point, tapi tidak cukup tetangga
- **Noise/Outlier**: tidak dekat dengan core point manapun (label = -1)

Keunggulan DBSCAN:
- Dapat menemukan cluster berbentuk **arbitrer** (tidak harus bulat)
- Otomatis mendeteksi **outlier** (noise)
- Tidak perlu menentukan $K$ cluster di awal


In [ ]:
# DBSCAN on moons dataset (K-Means fails here)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: K-Means vs DBSCAN on moons
km_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_km = km_moons.fit_predict(X_moons)

dbscan_moons = DBSCAN(eps=0.2, min_samples=5)
labels_db = dbscan_moons.fit_predict(X_moons)

axes[0,0].scatter(X_moons[:,0], X_moons[:,1], c=labels_km, cmap='Set1', s=20, alpha=0.8)
axes[0,0].scatter(km_moons.cluster_centers_[:,0], km_moons.cluster_centers_[:,1],
                   marker='*', s=250, c='black')
axes[0,0].set_title(f'K-Means K=2 on Moons\nSilhouette={silhouette_score(X_moons, labels_km):.3f}')

axes[0,1].scatter(X_moons[:,0], X_moons[:,1], c=labels_db, cmap='tab10', s=20, alpha=0.8)
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = np.sum(labels_db == -1)
axes[0,1].set_title(f'DBSCAN on Moons (eps=0.2)\nClusters={n_clusters}, Noise={n_noise}')

# DBSCAN eps sensitivity
for ax, eps in zip(axes[0,2:], [0.1]):
    db = DBSCAN(eps=eps, min_samples=5)
    lbls = db.fit_predict(X_moons)
    nc = len(set(lbls)) - (1 if -1 in lbls else 0)
    nn = np.sum(lbls == -1)
    ax.scatter(X_moons[:,0], X_moons[:,1], c=lbls, cmap='tab10', s=20, alpha=0.8)
    ax.set_title(f'DBSCAN eps={eps}\nClusters={nc}, Noise={nn}')

# Row 2: DBSCAN on Iris
X_iris_sc = StandardScaler().fit_transform(X_iris)
for ax, (eps, min_s) in zip(axes[1, :], [(0.5, 5), (0.8, 5), (1.0, 10)]):
    db = DBSCAN(eps=eps, min_samples=min_s)
    lbls = db.fit_predict(X_iris_sc)
    nc = len(set(lbls)) - (1 if -1 in lbls else 0)
    nn = np.sum(lbls == -1)
    ari = adjusted_rand_score(y_iris, np.where(lbls == -1, 3, lbls))
    ax.scatter(X_iris_sc[:,0], X_iris_sc[:,1], c=lbls, cmap='tab10', s=20, alpha=0.8)
    ax.set_title(f'DBSCAN on Iris (eps={eps}, min={min_s})\nClusters={nc}, Noise={nn}, ARI={ari:.3f}')

plt.tight_layout()
plt.show()


## 10.4 Hierarchical Clustering (Agglomerative)

Hierarchical clustering membangun hirarki cluster secara bottom-up (agglomerative) atau top-down (divisive).

**Agglomerative**: mulai dari setiap titik sebagai cluster terpisah, lalu merge cluster terdekat secara iteratif.

Linkage criteria (cara mengukur jarak antar cluster):
- **Ward**: minimasi variance dalam cluster; biasanya terbaik
- **Complete**: jarak maksimum antar dua cluster
- **Average**: jarak rata-rata antar titik di dua cluster
- **Single**: jarak minimum (rentan terhadap chaining effect)


In [ ]:
# Hierarchical clustering with dendrogram
X_iris_sc = StandardScaler().fit_transform(X_iris)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dendrogram
Z = linkage(X_iris_sc[:50], method='ward')  # use subset for clarity
dendrogram(Z, ax=axes[0], leaf_rotation=90, leaf_font_size=7)
axes[0].set_title('Hierarchical Clustering Dendrogram\n(Ward linkage, Iris first 50 samples)')
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Distance')
axes[0].axhline(y=3.5, color='red', linestyle='--', label='Cut at 3.5 → 3 clusters')
axes[0].legend()

# Agglomerative results
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg = agg.fit_predict(X_iris_sc)

ari_agg = adjusted_rand_score(y_iris, labels_agg)
axes[1].scatter(X_iris_sc[:,0], X_iris_sc[:,1], c=labels_agg, cmap='tab10', s=40, alpha=0.8)
axes[1].set_title(f'Agglomerative (Ward, K=3)\nARI vs True Labels: {ari_agg:.3f}')
axes[1].set_xlabel('Feature 1 (scaled)')
axes[1].set_ylabel('Feature 2 (scaled)')

plt.tight_layout()
plt.show()

# Compare linkage methods
print('=== Linkage Method Comparison (Iris, K=3) ===')
for linkage_m in ['ward', 'complete', 'average', 'single']:
    agg_m = AgglomerativeClustering(n_clusters=3, linkage=linkage_m)
    lbls = agg_m.fit_predict(X_iris_sc)
    ari = adjusted_rand_score(y_iris, lbls)
    sil = silhouette_score(X_iris_sc, lbls)
    db_score = davies_bouldin_score(X_iris_sc, lbls)
    print(f'{linkage_m:<10}: ARI={ari:.4f}, Silhouette={sil:.4f}, DB={db_score:.4f}')


## Ringkasan: Key Takeaways Chapter 10

| Method | K diperlukan? | Bentuk Cluster | Sensitif Outlier? | Terbaik Untuk |
|---|---|---|---|---|
| **K-Means** | Ya | Bulat (spherical) | Ya | Data besar, cluster bulat |
| **MiniBatch KMeans** | Ya | Bulat | Ya | Dataset sangat besar |
| **DBSCAN** | Tidak | Arbitrer | Tidak | Data dengan noise, cluster non-spherical |
| **Agglomerative** | Ya | Arbitrer | Tergantung linkage | Ketika struktur hierarkis penting |
| **Spectral** | Ya | Arbitrer | Ya | Data dengan cluster non-convex |

### Metrik Evaluasi Clustering

| Metrik | Range | Lebih Baik | Label Diperlukan? |
|---|---|---|---|
| **Silhouette Score** | [-1, 1] | Mendekati 1 | Tidak |
| **Davies-Bouldin Index** | [0, ∞) | Mendekati 0 | Tidak |
| **Adjusted Rand Index** | [-1, 1] | Mendekati 1 | Ya (ground truth) |
| **Inertia / WCSS** | [0, ∞) | Lebih kecil | Tidak |
